# DragonHPC ProcessGroup Tutorial for MPI Orchestration

**Estimated time:** ~40-55 minutes  
**Format:** 5 exercises, each with demo code, a short coding problem, and hidden solution.

---

## Session goals

This tutorial introduces DragonHPC ProcessGroup orchestration for Python
functions, serial executables, and MPI applications.

You will practice how to:

- build ProcessGroup jobs from ProcessTemplate objects
- launch MPI applications with PMI backends
- run several MPI applications at the same time
- manage stdout and output files
- control placement with Policy

The exercises are designed for all levels and focus on practical workflow
patterns commonly used in HPC automation.

---

## Setup - run this first

Start Jupyter from a Dragon-enabled environment (for example with
`dragon-jupyter`) and run the next code cell.

In [ ]:
import os
import re
import time
import queue
import socket
from pathlib import Path
from subprocess import Popen

import dragon
import multiprocessing as mp

from dragon.infrastructure.facts import PMIBackend
from dragon.infrastructure.policy import Policy
from dragon.native.process import Process, ProcessTemplate, MSG_PIPE, MSG_DEVNULL
from dragon.native.process_group import ProcessGroup

try:
    mp.set_start_method("dragon")
except RuntimeError:
    pass

print("Dragon ProcessGroup tutorial setup complete")

mpi_exe = Path("../dragon_native/mpi/mpi_hello").resolve()
print("mpi_hello path:", mpi_exe)
print("mpi_hello exists:", mpi_exe.exists())

If `mpi_hello` is not present, build it once:

```bash
cd ../dragon_native/mpi
make
```

Then rerun Cell 2 to confirm the executable exists.

---

## Exercise 1 - Orchestrate Python functions with ProcessGroup

**Background:**

`ProcessGroup` can launch many copies of a Python function template and
coordinate lifecycle through `init`, `start`, `join`, and `close`.

Demo pattern:

```python
def hello():
    print("hello from", os.getpid())

pg = ProcessGroup(restart=False)
pg.add_process(nproc=3, template=ProcessTemplate(target=hello))
pg.init()
pg.start()
pg.join()
pg.close()
```

**Your task:**

1. Write `ranked_hello(rank)` that prints rank, pid, and hostname
2. Build a ProcessGroup with 4 processes using one ProcessTemplate
3. Start, join, and close the group

In [ ]:
# -- Exercise 1 -- your code here -----------------------------------------------

def ranked_hello(rank):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def ranked_hello(rank):
    print(f"rank={rank} pid={os.getpid()} host={socket.gethostname()}")

pg = ProcessGroup(restart=False)
for rank in range(4):
    pg.add_process(nproc=1, template=ProcessTemplate(target=ranked_hello, args=(rank,)))

pg.init()
pg.start()
pg.join()
pg.close()
```

</details>

---

## Exercise 2 - Run a serial executable and manage I/O

**Background:**

`ProcessTemplate` supports output routing. You can capture one process output
to `MSG_PIPE` and write others to files or `DEVNULL`.

Demo pattern:

```python
pg = ProcessGroup(restart=False)
pg.add_process(nproc=1, template=ProcessTemplate(target="/bin/hostname", stdout=MSG_PIPE))
pg.init(); pg.start()
for puid in pg.puids:
    proc = Process(None, ident=puid)
    if proc.stdout_conn:
        print(proc.stdout_conn.recv())
pg.join(); pg.close()
```

**Your task:**

1. Create a ProcessGroup with two templates running `/bin/echo`
2. First template: one process, output to `MSG_PIPE`
3. Second template: three processes, output to files `echo_0.txt`, `echo_1.txt`, `echo_2.txt`
4. Print the piped output and then print one file's contents
5. Join and close the group

In [ ]:
# -- Exercise 2 -- your code here -----------------------------------------------

# Hint: target="/bin/echo", args=("hello from template",)

<details>
<summary><b>▶ Show Solution</b></summary>

```python
output_files = [f"echo_{i}.txt" for i in range(3)]
pg = ProcessGroup(restart=False)

pg.add_process(
    nproc=1,
    template=ProcessTemplate(target="/bin/echo", args=("hello from pipe",), stdout=MSG_PIPE),
)

for i in range(3):
    pg.add_process(
        nproc=1,
        template=ProcessTemplate(target="/bin/echo", args=(f"hello file {i}",), stdout=output_files[i]),
    )

pg.init()
pg.start()

for puid in pg.puids:
    proc = Process(None, ident=puid)
    if proc.stdout_conn:
        print("PIPE:", proc.stdout_conn.recv().strip())

pg.join()
pg.close()

with open(output_files[0], "r", encoding="utf-8") as f:
    print("FILE echo_0.txt:", f.read().strip())
```

</details>

---

## Exercise 3 - Launch an MPI application with ProcessGroup

**Background:**

To orchestrate an MPI executable, create a ProcessGroup with a PMI backend
and add process templates for rank groups. A common pattern is:

- rank 0 uses `MSG_PIPE` so we can parse output
- remaining ranks use `MSG_DEVNULL`

Demo pattern:

```python
exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)
pg.add_process(nproc=1, template=ProcessTemplate(target=exe, args=(), stdout=MSG_PIPE))
pg.add_process(nproc=3, template=ProcessTemplate(target=exe, args=(), stdout=MSG_DEVNULL))
pg.init(); pg.start(); pg.join(); pg.close()
```

**Your task:**

1. Launch `mpi_hello` with 4 total ranks using the two-template pattern
2. Capture and print output from the piped rank
3. Join and close the ProcessGroup

If `mpi_hello` is missing, build it first in `../dragon_native/mpi`.

In [ ]:
# -- Exercise 3 -- your code here -----------------------------------------------

# Hint: exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())

<details>
<summary><b>▶ Show Solution</b></summary>

```python
exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
if not Path(exe).exists():
    raise FileNotFoundError(f"Build first: {exe}")

pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)
pg.add_process(nproc=1, template=ProcessTemplate(target=exe, args=(), cwd=str(Path(exe).parent), stdout=MSG_PIPE))
pg.add_process(nproc=3, template=ProcessTemplate(target=exe, args=(), cwd=str(Path(exe).parent), stdout=MSG_DEVNULL))

pg.init()
pg.start()

for puid in pg.puids:
    proc = Process(None, ident=puid)
    if proc.stdout_conn:
        try:
            while True:
                line = proc.stdout_conn.recv()
                print(line.strip())
        except EOFError:
            pass

pg.join()
pg.close()
```

</details>

---

## Exercise 4 - Orchestrate several MPI jobs at the same time

**Background:**

HPC workflows often run multiple MPI applications concurrently. One practical
pattern is a producer function that creates and runs one ProcessGroup, then a
parent process launches multiple producers in parallel.

Demo pattern:

```python
def run_one_job(job_id, result_q):
    # create ProcessGroup for this job
    # start/join/close
    result_q.put((job_id, "done"))

result_q = mp.Queue()
p0 = Process(target=run_one_job, args=(0, result_q))
p1 = Process(target=run_one_job, args=(1, result_q))
p0.start(); p1.start()
print(result_q.get()); print(result_q.get())
p0.join(); p1.join()
```

**Your task:**

1. Write `run_mpi_job(job_id, num_ranks, result_q)` using ProcessGroup + `mpi_hello`
2. In each job, pipe rank 0 stdout and send one parsed line back via `result_q`
3. Launch two jobs concurrently with `num_ranks=4` and `num_ranks=6`
4. Print both results
5. Ensure both producer processes are joined

In [ ]:
# -- Exercise 4 -- your code here -----------------------------------------------

def run_mpi_job(job_id, num_ranks, result_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_mpi_job(job_id, num_ranks, result_q):
    exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
    pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)

    pg.add_process(
        nproc=1,
        template=ProcessTemplate(target=exe, args=(), cwd=str(Path(exe).parent), stdout=MSG_PIPE),
    )
    pg.add_process(
        nproc=max(0, num_ranks - 1),
        template=ProcessTemplate(target=exe, args=(), cwd=str(Path(exe).parent), stdout=MSG_DEVNULL),
    )

    pg.init()
    pg.start()

    first_line = None
    for puid in pg.puids:
        proc = Process(None, ident=puid)
        if proc.stdout_conn:
            try:
                first_line = proc.stdout_conn.recv().strip()
            except EOFError:
                first_line = "<no output>"
            break

    pg.join()
    pg.close()
    result_q.put((job_id, num_ranks, first_line))

result_q = mp.Queue()
p0 = Process(target=run_mpi_job, args=(0, 4, result_q))
p1 = Process(target=run_mpi_job, args=(1, 6, result_q))

p0.start()
p1.start()

print(result_q.get())
print(result_q.get())

p0.join()
p1.join()
```

</details>

---

## Exercise 5 - Placement control with Policy

**Background:**

Default placement spreads processes according to Dragon policy defaults
(often round-robin across available nodes). If you need explicit placement,
attach a `Policy` to ProcessGroup or ProcessTemplate.

Demo pattern:

```python
host = socket.gethostname()
local_policy = Policy(placement=Policy.Placement.HOST_NAME, host_name=host)
template = ProcessTemplate(target="/bin/hostname", policy=local_policy, stdout=MSG_PIPE)
```

**Your task:**

1. Write `placement_probe(out_q)` that reports `(pid, hostname)`
2. Build one `Policy` pinned to current hostname
3. Launch 4 processes in a ProcessGroup using that policy
4. Collect and print all reports
5. Print how many unique hostnames were used

In [ ]:
# -- Exercise 5 -- your code here -----------------------------------------------

def placement_probe(out_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def placement_probe(out_q):
    out_q.put((os.getpid(), socket.gethostname()))

q = mp.Queue()
host = socket.gethostname()
local_policy = Policy(placement=Policy.Placement.HOST_NAME, host_name=host)

pg = ProcessGroup(restart=False)
template = ProcessTemplate(target=placement_probe, args=(q,), policy=local_policy)
pg.add_process(nproc=4, template=template)

pg.init()
pg.start()

reports = [q.get() for _ in range(4)]
for item in reports:
    print(item)

pg.join()
pg.close()

hosts = sorted({host for _, host in reports})
print("unique hosts:", hosts)
print("count:", len(hosts))
```

</details>

---

## Summary

You used ProcessGroup as an orchestration primitive for Python and MPI jobs,
including concurrent launches, output handling, and placement control.

| Concept | API |
|---|---|
| Define work | `ProcessTemplate(...)` |
| Group orchestration | `ProcessGroup.add_process(...), init(), start(), join(), close()` |
| MPI launch backend | `ProcessGroup(..., pmi=PMIBackend.CRAY)` |
| Capture rank output | `stdout=MSG_PIPE`, `Process(None, ident=puid).stdout_conn` |
| Suppress extra output | `stdout=MSG_DEVNULL` |
| File output | `stdout="file.txt"` |
| Placement control | `Policy(...)` on group or template |

### Next steps

- Replace `mpi_hello` with your production MPI executable.
- Add parser processes to extract metrics from rank-0 stdout.
- Explore template-level and group-level policies for multi-job scheduling.